# metajax-regression — Worked Examples

This notebook walks through **seven progressive experiments** that demonstrate the full capability of the `metajax-regression` framework:

| # | Experiment | Model family | Search mode |
|---|---|---|---|
| 1 | Data loading & exploration | — | — |
| 2 | Fixed-effects Poisson baseline | Poisson | SA |
| 3 | Mixed NB structure search | Negative Binomial + random effects | SA |
| 4 | Zero-inflated model search | ZIP / ZINB | SA |
| 5 | Latent class search (2 classes) | LC-NB | SA |
| 6 | Membership variable search | LC-NB + membership equations | SA |
| 7 | Multi-objective Pareto search | NB + random effects | DE / HS |

---
> **Dataset used throughout**: `data/Ex-16-3.csv` — a road-segment crash-frequency panel.  
> Variables include `FREQ` (outcome), `LENGTH`, `AADT`, `FC`, `SPEED`, `URB`, `MEDIAN`, `SHOULDER`.

## 0 · Installation and imports

In [ ]:
# Install (uncomment if not already installed)
# !pip install metajax-regression

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

from metajax_regression import ExperimentBuilder

print('metajax-regression imported successfully ✅')

---
## 1 · Data loading and exploration

In [ ]:
# ------------------------------------------------------------------
# Load the dataset
# ------------------------------------------------------------------
df = pd.read_csv('data/Ex-16-3.csv')

# Engineer the exposure offset (site-years of 100 million vehicle-miles)
df['EXPOSE'] = df['LENGTH'] * df['AADT'] * 365 / 100_000_000

# The framework expects a column called OFFSET (log-scale offset in the linear predictor)
# We pass EXPOSE as the offset_col so the builder takes log(EXPOSE) internally.
# Using a zero-column here means no offset — swap to 'EXPOSE' to use the real one.
df['OFFSET'] = 0

# Rename the count outcome
df.rename(columns={'FREQ': 'Y'}, inplace=True)

print(f'Shape : {df.shape}')
df.head()

In [ ]:
# ------------------------------------------------------------------
# Quick outcome distribution check
# ------------------------------------------------------------------
y = df['Y']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(y, bins=range(int(y.max()) + 2), edgecolor='white', color='steelblue')
axes[0].set_xlabel('Crash count (Y)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Outcome distribution')

axes[1].scatter(df['AADT'], y, alpha=0.4, s=20, color='steelblue')
axes[1].set_xlabel('AADT')
axes[1].set_ylabel('Y (crashes)')
axes[1].set_title('Crashes vs AADT')

plt.tight_layout()
plt.show()

vr = y.var() / y.mean()
print(f'Mean  : {y.mean():.3f}')
print(f'Var   : {y.var():.3f}')
print(f'Var/Mean (dispersion index) : {vr:.3f}  — {"overdispersed → consider NB" if vr > 1.5 else "near-Poisson"}')

In [ ]:
# ------------------------------------------------------------------
# Build the ExperimentBuilder and call describe()
# describe() prints:
#   • outcome statistics + overdispersion index
#   • variable types, range, zero-rates
#   • full role-code guide
# ------------------------------------------------------------------
builder = ExperimentBuilder(
    df           = df,
    id_col       = 'ID',
    y_col        = 'Y',
    offset_col   = 'OFFSET',
    group_id_col = 'FC',        # functional class → enables grouped effects (role 4)
)

builder.describe()

In [ ]:
# ------------------------------------------------------------------
# suggest_config() gives per-variable role/distribution recommendations
# based on automatic type inference (binary, ordinal, count, continuous)
# ------------------------------------------------------------------
suggestions = builder.suggest_config(max_latent_classes=2)

# `suggestions` is a plain dict you can inspect or feed back in
print('\nReturned suggestion dict keys:', list(suggestions.keys()))

---
## 2 · Experiment 1 — Fixed-effects Poisson baseline

The simplest possible search: every variable is either **excluded (0)** or **fixed (1)**, Poisson dispersion, no random effects.  
This gives a pure fixed-effects count model and acts as a BIC benchmark.

In [ ]:
# ------------------------------------------------------------------
# Build evaluator — restrict role space to {0=Excluded, 1=Fixed}
# ------------------------------------------------------------------
eval_fixed = builder.build_evaluator(
    default_roles = [0, 1],     # only Excluded or Fixed
    mode          = 'single',   # minimise BIC
    R             = 50,         # draws not used for fixed models, but required
)

# ------------------------------------------------------------------
# Run Simulated Annealing
# max_iter is low for a notebook demo — increase to 2000+ in practice
# ------------------------------------------------------------------
result_fixed = builder.run(
    evaluator = eval_fixed,
    algo      = 'sa',
    max_iter  = 500,
    seed      = 42,
)

bic_baseline = result_fixed['best_score']
print(f'\nBaseline fixed-effects BIC : {bic_baseline:.2f}')

---
## 3 · Experiment 2 — Mixed Negative Binomial structure search

Open the role space to include **random independent (2)** and **random correlated (3)** effects and allow **Negative Binomial** dispersion.  
The search finds the combination of variable roles and distributions that minimises BIC.

In [ ]:
# ------------------------------------------------------------------
# Build evaluator for mixed NB search
# ------------------------------------------------------------------
eval_mixed = builder.build_evaluator(
    default_roles = [0, 1, 2, 3],    # Excluded / Fixed / Rnd-Ind / Rnd-Cor
    mode          = 'single',
    R             = 150,             # more draws → better integral approximation
    # Force AADT to always enter the model (either fixed or random)
    fixed_override = {
        'AADT': [1, 2, 3],           # cannot be excluded
    },
)

result_mixed = builder.run(
    evaluator = eval_mixed,
    algo      = 'sa',
    max_iter  = 1000,
    seed      = 42,
    alpha     = 0.995,               # SA cooling rate
    n_starts  = 1,
)

bic_mixed = result_mixed['best_score']
print(f'\nMixed NB BIC    : {bic_mixed:.2f}')
print(f'Fixed baseline  : {bic_baseline:.2f}')
print(f'Improvement     : {bic_baseline - bic_mixed:.2f}')

In [ ]:
# ------------------------------------------------------------------
# Inspect the search trajectory
# result['scores'] holds the BIC of every accepted solution
# ------------------------------------------------------------------
scores = result_mixed['scores']

plt.figure(figsize=(10, 4))
plt.plot(scores, lw=0.8, color='steelblue', alpha=0.8)
plt.axhline(bic_mixed, color='crimson', lw=1.5, ls='--', label=f'Best BIC={bic_mixed:.1f}')
plt.xlabel('SA iteration')
plt.ylabel('BIC')
plt.title('Simulated Annealing convergence — Mixed NB')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4 · Experiment 3 — Zero-inflated model search

When a count dataset has excess zeros beyond what Poisson or NB predicts, a **zero-inflation** component may help.  
Role **6** allows a variable to enter the zero-inflation probability equation.

In [ ]:
# ------------------------------------------------------------------
# Check zero-inflation severity
# ------------------------------------------------------------------
zero_rate = (df['Y'] == 0).mean()
print(f'Zero rate in outcome: {zero_rate*100:.1f}%')
if zero_rate > 0.15:
    print('High zero rate → zero-inflation roles (6) are worth exploring.')
else:
    print('Relatively low zero rate → ZI may not improve BIC significantly.')

In [ ]:
# ------------------------------------------------------------------
# Build evaluator with ZI role (6) enabled
# ------------------------------------------------------------------
eval_zi = builder.build_evaluator(
    default_roles = [0, 1, 2, 6],   # Excluded / Fixed / Rnd-Ind / ZI
    mode          = 'single',
    R             = 150,
    # Allow specific variables to be ZI only (e.g. road-type indicator)
    fixed_override = {
        'URB':  [0, 1, 6],          # URB can be Fixed or ZI
        'AADT': [1, 2],             # AADT stays in outcome equation
    },
)

result_zi = builder.run(
    evaluator = eval_zi,
    algo      = 'sa',
    max_iter  = 1000,
    seed      = 0,
)

bic_zi = result_zi['best_score']
print(f'\nZI model BIC    : {bic_zi:.2f}')
print(f'Mixed NB BIC    : {bic_mixed:.2f}')

---
## 5 · Experiment 4 — Latent class search (up to 2 classes)

**Latent class (LC) models** assume the population consists of *C* unobserved sub-groups, each with its own parameter vector.  
Setting `max_latent_classes=2` tells the search to try both 1-class and 2-class specifications.  
The LC variant is warm-started from the 1-class fit for stability.

> BIC penalises the extra parameters of multi-class models, so a 2-class solution will only "win" if the data strongly support it.

In [ ]:
# ------------------------------------------------------------------
# Build evaluator — LC search, no membership variables yet
# ------------------------------------------------------------------
eval_lc = builder.build_evaluator(
    mode               = 'single',
    max_latent_classes = 2,      # search over 1 and 2 classes
    R                  = 150,
)

print('Decision vector dimension:', 2 * len(eval_lc.vars) + 2)
print('  (2 × n_vars + dispersion_bit + lc_code)')

In [ ]:
result_lc = builder.run(
    evaluator = eval_lc,
    algo      = 'sa',
    max_iter  = 1500,
    seed      = 0,
)

bic_lc = result_lc['best_score']
best   = result_lc['best_solution']

# Decode the number of latent classes in the best solution
D       = len(eval_lc.vars)
lc_code = int(best[2*D + 1])
n_cls   = (lc_code % eval_lc.max_latent_classes) + 1

print(f'\nBest BIC         : {bic_lc:.2f}')
print(f'Latent classes   : {n_cls}')
print(f'Mixed NB BIC     : {bic_mixed:.2f}')
print(f'Δ BIC (LC − NB)  : {bic_lc - bic_mixed:.2f}')

---
## 6 · Experiment 5 — Membership variable search

When `max_latent_classes > 1` you can designate variables to the **class-membership equation** using roles 7 and 8.

| Role | Effect on membership | Effect on outcome |
|---|---|---|
| 7 | ✅ Shifts class probabilities | ❌ None |
| 8 | ✅ Shifts class probabilities | ✅ Fixed (class-specific β) |

Here we allow `URB` and `SPEED` as membership variables — sensible because road urbanisation and speed limit may determine the *type* of crash environment a site falls into.

In [ ]:
# ------------------------------------------------------------------
# membership_override lets you add roles 7 or 8 for specific variables
# ------------------------------------------------------------------
eval_mem = builder.build_evaluator(
    mode               = 'single',
    max_latent_classes = 2,
    R                  = 150,
    membership_override = {
        'URB':   [0, 7],        # can be excluded OR membership-only
        'SPEED': [0, 7, 8],     # can be excluded, membership-only, OR membership+fixed
    },
)

result_mem = builder.run(
    evaluator = eval_mem,
    algo      = 'sa',
    max_iter  = 2000,
    seed      = 1,
)

bic_mem  = result_mem['best_score']
best_mem = result_mem['best_solution']

# Count membership-role variables in the best solution
D       = len(eval_mem.vars)
n_mem7  = sum(1 for i in range(D) if int(best_mem[i]) == 7)
n_mem8  = sum(1 for i in range(D) if int(best_mem[i]) == 8)
lc_cls  = (int(best_mem[2*D+1]) % eval_mem.max_latent_classes) + 1

print(f'\nBIC (membership LC) : {bic_mem:.2f}')
print(f'BIC (no membership) : {bic_lc:.2f}')
print(f'Latent classes      : {lc_cls}')
print(f'Membership-only (7) : {n_mem7} variable(s)')
print(f'Membership+fixed (8): {n_mem8} variable(s)')

In [ ]:
# ------------------------------------------------------------------
# Print role assignments for the best solution
# ------------------------------------------------------------------
_ROLE_NAMES = {
    0: 'Excluded', 1: 'Fixed',      2: 'Rnd-Ind',  3: 'Rnd-Cor',
    4: 'Grouped',  5: 'Hetero',     6: 'ZI',       7: 'Membership',
    8: 'Mem+Fixed',
}

print('\nBest variable assignments:')
print(f'{"Variable":<20} {"Role"}')
print('-' * 35)
for i, var in enumerate(eval_mem.vars):
    role = int(best_mem[i])
    if role != 0:
        print(f'{var:<20} {_ROLE_NAMES[role]}')

---
## 7 · Experiment 6 — 3-class latent structure with functional class

This experiment tests whether the **functional class (FC)** variable — which has 5 categories — can be captured through latent classes.  
Setting `max_latent_classes=3` allows up to 3 unobserved sub-populations.  
FC is forced to role 7 or 8, so the search directly tests whether class membership reproduces the FC structure.

In [ ]:
print('Functional class (FC) distribution:')
print(df['FC'].value_counts().sort_index().to_string())

In [ ]:
# ------------------------------------------------------------------
# Attempt to capture FC through latent class membership
# ------------------------------------------------------------------
eval_fc_lc = builder.build_evaluator(
    mode               = 'single',
    max_latent_classes = 3,
    R                  = 200,
    membership_override = {
        # Force FC into the membership equation only
        # (the algorithm will decide 7 vs 8)
        'FC': [7, 8],
        # Allow URB and SPEED to also influence membership
        'URB':   [0, 7, 8],
        'SPEED': [0, 7, 8],
    },
    # Remove FC from the grouped-effect role since we're using it as membership
    fixed_override = {
        'AADT': [1, 2, 3],   # keep AADT always active
    },
)

result_fc = builder.run(
    evaluator = eval_fc_lc,
    algo      = 'sa',
    max_iter  = 2000,
    seed      = 7,
    alpha     = 0.998,       # slower cooling for 3-class search
)

bic_fc  = result_fc['best_score']
best_fc = result_fc['best_solution']
D       = len(eval_fc_lc.vars)
lc_fc   = (int(best_fc[2*D+1]) % eval_fc_lc.max_latent_classes) + 1

print(f'\nBIC (FC membership LC) : {bic_fc:.2f}')
print(f'Latent classes found   : {lc_fc}')

---
## 8 · Experiment 7 — Multi-objective search (BIC + test RMSE Pareto front)

In `mode='multi'` the framework simultaneously minimises **BIC** and **test-set RMSE**.  
The result is a **Pareto front** of non-dominated solutions — you can trade off parsimony (low BIC) against predictive accuracy (low RMSE) by picking the point that best matches your use case.

In [ ]:
# ------------------------------------------------------------------
# Multi-objective evaluator
# ------------------------------------------------------------------
eval_multi = builder.build_evaluator(
    mode               = 'multi',
    max_latent_classes = 2,
    R                  = 150,
    default_roles      = [0, 1, 2, 3],
)

# Run Adaptive Differential Evolution + NSGA-II
result_multi = builder.run(
    evaluator       = eval_multi,
    algo            = 'de',
    max_iter        = 1000,
    seed            = 0,
    population_size = 20,
    F               = 0.5,
    CR              = 0.7,
)

In [ ]:
# ------------------------------------------------------------------
# Plot the Pareto front
# result_multi['pareto_scores'] is (n_solutions, 2): [BIC, test_RMSE]
# ------------------------------------------------------------------
if 'pareto_scores' in result_multi:
    pf = np.array(result_multi['pareto_scores'])
    
    plt.figure(figsize=(8, 5))
    plt.scatter(pf[:, 0], pf[:, 1], c='steelblue', s=60, zorder=3)
    
    # Sort by BIC for the line
    idx = np.argsort(pf[:, 0])
    plt.plot(pf[idx, 0], pf[idx, 1], 'steelblue', lw=1, alpha=0.5)
    
    plt.xlabel('BIC (parsimony ↓)')
    plt.ylabel('Test RMSE (accuracy ↓)')
    plt.title('Pareto front — BIC vs Test RMSE')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Pareto scores not returned in this result format — check algo output.')

---
## 9 · BIC comparison across all experiments

In [ ]:
# ------------------------------------------------------------------
# Collect results (replace with your actual values)
# ------------------------------------------------------------------
results_summary = {
    'Fixed Poisson (baseline)':  bic_baseline,
    'Mixed NB':                  bic_mixed,
    'Zero-Inflated':             bic_zi,
    'Latent Class (no mem)':     bic_lc,
    'LC + Membership vars':      bic_mem,
    'LC + FC membership (3-cls)': bic_fc,
}

# Table
print(f'{"Model":<35} {"BIC":>12} {"ΔBIC vs baseline":>18}')
print('-' * 68)
for name, bic in results_summary.items():
    delta = bic - bic_baseline
    marker = '  ← best' if bic == min(results_summary.values()) else ''
    print(f'{name:<35} {bic:>12.2f} {delta:>+18.2f}{marker}')

In [ ]:
# ------------------------------------------------------------------
# Bar chart
# ------------------------------------------------------------------
names = list(results_summary.keys())
bics  = list(results_summary.values())
best_bic = min(bics)

colors = ['#2196F3' if b > best_bic else '#F44336' for b in bics]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(names, bics, color=colors, edgecolor='white')

for bar, bic in zip(bars, bics):
    ax.text(bic + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{bic:.1f}', va='center', fontsize=9)

ax.set_xlabel('BIC (lower is better)')
ax.set_title('Model comparison — BIC across experiments')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 10 · Tips and best practices

### Choosing `R` (Halton draws)

| Use case | Recommended R |
|---|---|
| Quick exploration, binary/fixed-only | 50–100 |
| Mixed models (1–2 random effects) | 150–200 |
| Many random effects or correlated | 300–500 |
| Latent class models | 150–300 |

Higher R → better integral approximation, but slower per-evaluation.

### Choosing `max_iter`

| Search space | Recommended iterations |
|---|---|
| ≤5 variables, fixed-only | 500 |
| 5–10 variables, mixed | 1 000–2 000 |
| 10+ variables, LC | 3 000–5 000 |

### `fixed_override` patterns

```python
# Force a variable to always be in the model as fixed:
fixed_override = {'AADT': [1]}

# Restrict to a subset of roles but keep flexibility:
fixed_override = {'SPEED': [0, 1, 2]}   # no correlated randoms

# Combine with membership_override (membership_override takes priority):
membership_override = {'URB': [0, 7, 8]}
```

### When to use multi-objective (`mode='multi'`)

Use `mode='multi'` when:
- You need to deliver a model to a user who cares about **prediction accuracy** (not just statistical fit)
- You have enough data for a reliable test split
- You want to visualise the **parsimony–accuracy trade-off**

Stick with `mode='single'` when:
- Your primary goal is **statistical inference** (coefficient estimates, significance)
- Computation time is limited
- Small datasets (test split is noisy)

### Algorithm choice

```python
# Single-objective → always use SA
result = builder.run(evaluator, algo='sa', max_iter=3000)

# Multi-objective → DE or HS
result = builder.run(evaluator, algo='de', max_iter=2000, population_size=30)
result = builder.run(evaluator, algo='hs', max_iter=2000, population_size=30)
```

---
## 11 · Loading your own dataset

Replace the data-loading block below with your own file path and column names.

In [ ]:
# ------------------------------------------------------------------
# Template: load your own panel count dataset
# ------------------------------------------------------------------

# df = pd.read_csv('path/to/your_data.csv')

# Required column roles:
#   id_col       — unique panel unit identifier (site, subject, entity)
#   y_col        — non-negative integer count outcome
#   offset_col   — (optional) log-offset; use a zero column if not needed
#   group_id_col — (optional) group membership for grouped random effects

# Example column engineering:
# df['EXPOSURE'] = df['LENGTH'] * df['VOLUME'] * 365 / 1e8
# df['OFFSET']   = np.log(df['EXPOSURE'] + 1e-9)   # ← this becomes the log-offset

# my_builder = ExperimentBuilder(
#     df           = df,
#     id_col       = 'SITE_ID',
#     y_col        = 'CRASHES',
#     offset_col   = 'OFFSET',
#     group_id_col = 'ROAD_CLASS',
# )

# my_builder.describe()
# my_builder.suggest_config(max_latent_classes=2)

print('Template ready — uncomment and fill in your details.')